# MSMARCO 1M Learned RARS Router

This notebook upgrades the earlier query-adaptive RARS diagnostics from simple
single-feature thresholds to a held-out learned query-level router.

Goal:

```text
For each query, choose the cheapest correction depth among Top0 / Top20 / Top40
while preserving qrels retrieval quality.
```

The notebook is intentionally conservative:

- It evaluates 5-fold held-out routing, not only oracle routing.
- It reports fixed-depth baselines, oracle routing, and learned routers together.
- It optimizes a query-level decision, not a new sidecar representation.
- It uses the already-computed RARS score-error weighted sidecar from the
  MS MARCO 1M RARS notebook.

Expected prerequisite:

```text
notebooks/MSMARCO_1M_Retrieval_Aware_Residual_Basis.ipynb
```

should have already produced the current candidate cache, exact candidate scores,
basis files, and int8 sidecar codes under:

```text
/content/gdrive/MyDrive/rag-pq-checkpoints/retrieval_aware_residual_basis_msmarco_1m
```

In [ ]:
from google.colab import drive

drive.mount("/content/gdrive", force_remount=True)

## 1. Imports, paths, and configuration

In [ ]:
from pathlib import Path
import json
import math
import hashlib
import warnings

import numpy as np
import pandas as pd

try:
    import faiss
except Exception:
    !pip -q install faiss-gpu-cu12 || pip -q install faiss-gpu || pip -q install faiss-cpu
    import faiss

try:
    from sklearn.model_selection import KFold, StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    from sklearn.metrics import accuracy_score, balanced_accuracy_score
except Exception:
    !pip -q install scikit-learn
    from sklearn.model_selection import KFold, StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    from sklearn.metrics import accuracy_score, balanced_accuracy_score

ROOT = Path("/content/gdrive/MyDrive/rag-pq-checkpoints")

GATE0 = ROOT / "msmarco_basis_gate0_cache"
GATE3 = ROOT / "msmarco_1m_pq_residual_gate3"

RARS_DIR = ROOT / "retrieval_aware_residual_basis_msmarco_1m"
CURRENT_CACHE_DIR = RARS_DIR / "current_1m_candidate_cache"

OUT_DIR = RARS_DIR / "learned_rars_router"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOC_IDS_PATH = GATE0 / "doc_ids.int64.memmap"
QUERY_PATH = GATE0 / "query_vectors.fp32.npy"
EVAL_QIDS_PATH = GATE0 / "eval_qids.json"
HELDOUT_SPLIT_PATH = GATE3 / "heldout_eval_split.json"

ANN_ROWS_PATH = CURRENT_CACHE_DIR / "ivfpq_m32_nprobe16_top100_internal_rows.npy"
ANN_SCORES_PATH = CURRENT_CACHE_DIR / "ivfpq_m32_nprobe16_top100_scores.npy"
EXACT_SCORES_PATH = CURRENT_CACHE_DIR / "exact_scores_for_current_candidates.npy"

BASIS_NAME = "score_error_weighted"
BASIS_PATH = RARS_DIR / "basis_score_error_weighted_rank16_internal.npy"
CODES_PATH = RARS_DIR / "codes_score_error_weighted_rank16.int8.memmap"
SCALES_PATH = RARS_DIR / "scales_score_error_weighted_rank16.float32.npy"

N_DOCS = 1_000_000
DIM = 384
RANK = 16
TOP_L = 100
FINAL_K = 10

DEPTHS = np.array([0, 20, 40], dtype=np.int64)
ALPHA = 0.75
N_SPLITS = 5
RANDOM_STATE = 42

CONFIG = {
    "package": "learned_rars_router_msmarco_1m",
    "dataset": "MS MARCO deterministic 1M passage subset",
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "base_index": "frozen IVF-PQ M=32 nlist=512 nprobe=16",
    "candidate_pool": int(TOP_L),
    "depths": DEPTHS.tolist(),
    "basis": BASIS_NAME,
    "alpha": float(ALPHA),
    "sidecar_rank": int(RANK),
    "router_protocol": f"{N_SPLITS}-fold held-out query-level routing",
    "random_state": int(RANDOM_STATE),
}

with open(OUT_DIR / "router_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

for path in [
    DOC_IDS_PATH,
    QUERY_PATH,
    EVAL_QIDS_PATH,
    HELDOUT_SPLIT_PATH,
    ANN_ROWS_PATH,
    ANN_SCORES_PATH,
    EXACT_SCORES_PATH,
    BASIS_PATH,
    CODES_PATH,
    SCALES_PATH,
]:
    print(path, "exists =", path.exists())

print("faiss version:", faiss.__version__)
print("output:", OUT_DIR)

## 2. Load held-out queries, candidates, and RARS sidecar

In [ ]:
# Load artifacts and rebuild missing exact candidate scores if needed.
# This cell is robust to missing exact_scores cache and does not require CACHE_DIR.

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def first_existing(paths, name):
    paths = [Path(p) for p in paths]
    for p in paths:
        if p.exists():
            print(f"{name}: {p}")
            return p

    raise FileNotFoundError(
        f"Could not find {name}. Tried:\n" + "\n".join(str(p) for p in paths)
    )


def discover_first(root, patterns, name, exclude_dirs=None):
    root = Path(root)
    exclude_dirs = set(exclude_dirs or [])
    hits = []

    for pat in patterns:
        for p in sorted(root.rglob(pat)):
            if not p.is_file():
                continue
            if any(part in exclude_dirs for part in p.parts):
                continue
            hits.append(p)

    # De-duplicate while preserving order.
    seen = set()
    uniq = []
    for p in hits:
        if p not in seen:
            uniq.append(p)
            seen.add(p)

    if uniq:
        print(f"{name} discovered:", uniq[0])
        return uniq[0]

    raise FileNotFoundError(
        f"Could not discover {name} under {root} with patterns: {patterns}"
    )


# ---------------------------------------------------------------------
# Core metadata and candidate cache
# ---------------------------------------------------------------------

doc_ids = np.memmap(DOC_IDS_PATH, dtype=np.int64, mode="r", shape=(N_DOCS,))
eval_qids_all = load_json(EVAL_QIDS_PATH)
heldout_split = load_json(HELDOUT_SPLIT_PATH)

eval_query_rows = np.asarray(heldout_split["query_rows"], dtype=np.int64)
eval_query_ids = [str(x) for x in heldout_split["query_ids"]]

ann_internal_rows = np.load(ANN_ROWS_PATH).astype(np.int64)
ann_scores = np.load(ANN_SCORES_PATH).astype(np.float32)

assert ann_internal_rows.shape == (len(eval_query_rows), TOP_L), ann_internal_rows.shape
assert ann_scores.shape == ann_internal_rows.shape, ann_scores.shape


# ---------------------------------------------------------------------
# Query embeddings
# ---------------------------------------------------------------------

QUERY_CANDIDATES = [
    QUERY_PATH,
    GATE0 / "query_embeddings.npy",
    GATE0 / "queries.npy",
    GATE0 / "Q.npy",
    GATE0 / "msmarco_query_embeddings.npy",
    GATE0 / "dev_query_embeddings.npy",
    ROOT / "query_embeddings.npy",
    ROOT / "queries.npy",
    ROOT / "Q.npy",
]

try:
    query_path = first_existing(QUERY_CANDIDATES, "query embeddings")
except FileNotFoundError:
    query_path = discover_first(
        ROOT,
        patterns=[
            "*query*embedding*.npy",
            "*query*vector*.npy",
            "*queries*.npy",
            "*Q*.npy",
        ],
        name="query embeddings",
        exclude_dirs={"learned_rars_router"},
    )

Q_all = np.load(query_path, mmap_mode="r").astype(np.float32)

# The candidate cache is for the held-out query split.
# If Q_all is already held-out only, use it directly.
# If Q_all contains all dev queries, select eval_query_rows.
if Q_all.shape[0] == len(eval_query_rows):
    Q_eval = np.asarray(Q_all, dtype=np.float32)
    print("Q_eval already heldout:", Q_eval.shape)
elif Q_all.shape[0] > int(eval_query_rows.max()):
    Q_eval = np.asarray(Q_all[eval_query_rows], dtype=np.float32)
    print("Q_all:", Q_all.shape)
    print("Q_eval selected by eval_query_rows:", Q_eval.shape)
else:
    raise ValueError(
        f"Cannot align query embeddings with heldout rows: "
        f"Q_all.shape={Q_all.shape}, max eval_query_row={int(eval_query_rows.max())}"
    )

assert Q_eval.shape[0] == ann_internal_rows.shape[0], Q_eval.shape
assert Q_eval.shape[1] == DIM, Q_eval.shape


# ---------------------------------------------------------------------
# Exact candidate scores
# ---------------------------------------------------------------------

if EXACT_SCORES_PATH.exists():
    exact_scores = np.load(EXACT_SCORES_PATH).astype(np.float32)
    print("loaded exact_scores:", EXACT_SCORES_PATH, exact_scores.shape)
else:
    print("missing exact_scores cache, rebuilding from Q_eval and document embeddings...")
    print("expected path:", EXACT_SCORES_PATH)

    DOC_EMB_CANDIDATES = [
        GATE0 / "doc_embeddings.npy",
        GATE0 / "document_embeddings.npy",
        GATE0 / "corpus_embeddings.npy",
        GATE0 / "passage_embeddings.npy",
        GATE0 / "X.npy",
        ROOT / "doc_embeddings.npy",
        ROOT / "document_embeddings.npy",
        ROOT / "corpus_embeddings.npy",
        ROOT / "passage_embeddings.npy",
        ROOT / "X.npy",
    ]

    try:
        doc_emb_path = first_existing(DOC_EMB_CANDIDATES, "document embeddings")
    except FileNotFoundError:
        doc_emb_path = discover_first(
            ROOT,
            patterns=[
                "*doc*embedding*.npy",
                "*document*embedding*.npy",
                "*corpus*embedding*.npy",
                "*passage*embedding*.npy",
                "*X*.npy",
            ],
            name="document embeddings",
            exclude_dirs={"learned_rars_router", "current_1m_candidate_cache"},
        )

    X_doc = np.load(doc_emb_path, mmap_mode="r")

    assert X_doc.shape[0] == N_DOCS, (X_doc.shape, N_DOCS)
    assert X_doc.shape[1] == DIM, (X_doc.shape, DIM)

    exact_scores = np.empty_like(ann_scores, dtype=np.float32)

    batch_size = 64
    for start in range(0, len(Q_eval), batch_size):
        end = min(start + batch_size, len(Q_eval))

        rows_b = ann_internal_rows[start:end]              # [B, L]
        q_b = Q_eval[start:end].astype(np.float32)         # [B, D]
        x_b = np.asarray(X_doc[rows_b], dtype=np.float32)  # [B, L, D]

        exact_scores[start:end] = np.einsum("bd,bld->bl", q_b, x_b)

        if start % 512 == 0:
            print(f"rebuilt exact scores: {end}/{len(Q_eval)}")

    EXACT_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(EXACT_SCORES_PATH, exact_scores)
    print("rebuilt and saved exact_scores:", EXACT_SCORES_PATH, exact_scores.shape)

assert exact_scores.shape == ann_internal_rows.shape, exact_scores.shape


# ---------------------------------------------------------------------
# RARS basis and sidecar codes
# ---------------------------------------------------------------------

B = np.load(BASIS_PATH).astype(np.float32)
scales = np.load(SCALES_PATH).astype(np.float32)
codes = np.memmap(CODES_PATH, dtype=np.int8, mode="r", shape=(N_DOCS, RANK))

assert B.shape == (DIM, RANK), B.shape
assert scales.shape == (RANK,), scales.shape
assert codes.shape == (N_DOCS, RANK), codes.shape


# ---------------------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------------------

for i in range(min(10, len(eval_query_rows))):
    qrow = int(eval_query_rows[i])
    assert str(eval_qids_all[qrow]) == str(eval_query_ids[i])

candidate_doc_ids = np.asarray(doc_ids[ann_internal_rows], dtype=np.int64)

print("queries:", len(eval_query_rows))
print("ann_internal_rows:", ann_internal_rows.shape)
print("ann_scores:", ann_scores.shape)
print("exact_scores:", exact_scores.shape)
print("candidate_doc_ids:", candidate_doc_ids.shape)
print("Q_eval:", Q_eval.shape)
print("B:", B.shape)
print("codes:", codes.shape)
print("scales:", scales.shape)


## 3. Load qrels robustly

In [ ]:
def parse_qrels_tsv(path):
    qrels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("query"):
                continue
            parts = line.replace(",", "\t").split()
            if len(parts) < 2:
                continue

            if len(parts) >= 4 and parts[1] in {"0", "Q0"}:
                qid, docid = parts[0], parts[2]
                rel = float(parts[3])
            elif len(parts) >= 3:
                qid, docid = parts[0], parts[1]
                try:
                    rel = float(parts[2])
                except Exception:
                    rel = 1.0
            else:
                qid, docid, rel = parts[0], parts[1], 1.0

            if rel > 0:
                qrels.setdefault(str(qid), set()).add(int(docid))
    return qrels


def parse_qrels_json(path):
    data = load_json(path)
    qrels = {}

    if isinstance(data, dict):
        for qid, value in data.items():
            if isinstance(value, dict):
                rel_docs = [int(docid) for docid, rel in value.items() if float(rel) > 0]
            elif isinstance(value, list):
                rel_docs = [int(x) for x in value]
            else:
                continue
            qrels[str(qid)] = set(rel_docs)

    elif isinstance(data, list):
        for row in data:
            if not isinstance(row, dict):
                continue
            qid = row.get("query_id", row.get("qid", row.get("query")))
            docid = row.get("doc_id", row.get("docid", row.get("pid", row.get("passage_id"))))
            rel = row.get("relevance", row.get("score", row.get("rel", 1)))
            if qid is not None and docid is not None and float(rel) > 0:
                qrels.setdefault(str(qid), set()).add(int(docid))

    return qrels


def qrels_from_heldout_split(heldout):
    qrels = {}
    possible_keys = [
        "qrels",
        "relevant_doc_ids",
        "positive_doc_ids",
        "positives",
        "query_positive_doc_ids",
    ]

    for key in possible_keys:
        if key not in heldout:
            continue
        value = heldout[key]

        if isinstance(value, dict):
            for qid, docs in value.items():
                if isinstance(docs, dict):
                    docs = [d for d, rel in docs.items() if float(rel) > 0]
                qrels[str(qid)] = set(int(d) for d in docs)
            return qrels

        if isinstance(value, list) and len(value) == len(eval_query_ids):
            for qid, docs in zip(eval_query_ids, value):
                if docs is None:
                    docs = []
                qrels[str(qid)] = set(int(d) for d in docs)
            return qrels

    return qrels


def find_and_load_qrels():
    q = qrels_from_heldout_split(heldout_split)
    if q:
        print("loaded qrels from heldout_split")
        return q, "heldout_split"

    candidates = []
    for root in [GATE0, GATE3, RARS_DIR, ROOT]:
        if not root.exists():
            continue
        for pattern in [
            "*qrels*.tsv",
            "*qrels*.txt",
            "*qrels*.json",
            "*qrel*.tsv",
            "*qrel*.txt",
            "*qrel*.json",
            "*dev*.tsv",
        ]:
            candidates.extend(sorted(root.glob(pattern)))

    candidates = sorted(
        set(candidates),
        key=lambda p: ("qrel" not in p.name.lower(), len(str(p))),
    )

    errors = []
    for path in candidates:
        try:
            if path.suffix.lower() == ".json":
                q = parse_qrels_json(path)
            else:
                q = parse_qrels_tsv(path)

            overlap = sum(1 for qid in eval_query_ids if str(qid) in q)
            if overlap > 0:
                print("loaded qrels:", path)
                print("heldout qid coverage:", overlap, "/", len(eval_query_ids))
                return q, str(path)
        except Exception as e:
            errors.append((str(path), repr(e)))

    print("qrels candidates tried:", len(candidates))
    for item in errors[:10]:
        print("qrels parse error:", item)
    raise FileNotFoundError(
        "Could not find qrels for held-out MS MARCO queries. "
        "Place a qrels TSV/JSON file under GATE0, GATE3, RARS_DIR, or ROOT, "
        "or add positives to heldout_eval_split.json."
    )


qrels_by_qid, qrels_source = find_and_load_qrels()

heldout_rel_sets = []
missing = []
for qid in eval_query_ids:
    rel = set(int(x) for x in qrels_by_qid.get(str(qid), set()))
    heldout_rel_sets.append(rel)
    if not rel:
        missing.append(str(qid))

print("qrels source:", qrels_source)
print("queries with at least one relevant doc:", len(heldout_rel_sets) - len(missing), "/", len(heldout_rel_sets))
print("missing qrels:", len(missing))

if len(missing) > 0:
    warnings.warn("Some held-out queries have no qrels; their per-query metrics will be zero.")

## 4. Compute correction matrices for Top0 / Top20 / Top40

In [ ]:
def get_eval_query(qi):
    return Q_eval[int(qi)].astype(np.float32)


def compute_correction_matrix(B, codes, scales, top_b=40, batch_log_every=100):
    corr = np.zeros_like(ann_scores, dtype=np.float32)

    for qi in range(ann_internal_rows.shape[0]):
        q = get_eval_query(qi)
        q_proj = q @ B

        ids = ann_internal_rows[qi, :top_b]
        coeff = codes[ids].astype(np.float32) * scales[None, :]
        corr[qi, :top_b] = (coeff @ q_proj).astype(np.float32)

        if batch_log_every and (qi + 1) % batch_log_every == 0:
            print("correction", qi + 1, "/", ann_internal_rows.shape[0])

    return corr


CORR40_PATH = OUT_DIR / f"correction_{BASIS_NAME}_alpha{ALPHA}_top40.npy"

if CORR40_PATH.exists():
    correction_alpha = np.load(CORR40_PATH).astype(np.float32)
    print("loaded:", CORR40_PATH)
else:
    correction_raw = compute_correction_matrix(B, codes, scales, top_b=40)
    correction_alpha = (ALPHA * correction_raw).astype(np.float32)
    np.save(CORR40_PATH, correction_alpha)
    print("saved:", CORR40_PATH)

assert correction_alpha.shape == ann_scores.shape, correction_alpha.shape

score_by_depth = {}
rank_by_depth = {}

for depth in DEPTHS:
    corrected = ann_scores.copy()
    if depth > 0:
        corrected[:, :depth] += correction_alpha[:, :depth]
    score_by_depth[int(depth)] = corrected
    rank_by_depth[int(depth)] = np.argsort(-corrected, axis=1)

print("depths ready:", list(score_by_depth.keys()))


## 5. Qrels metrics by query and fixed-depth baselines

In [ ]:
def dcg_at_k(binary_rels):
    binary_rels = np.asarray(binary_rels, dtype=np.float32)
    if binary_rels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, binary_rels.size + 2))
    return float(np.sum(binary_rels * discounts))


def per_query_metrics_from_order(qi, order, top_k=10):
    rel = heldout_rel_sets[qi]
    if not rel:
        return {
            "recall@10": 0.0,
            "success@10": 0.0,
            "mrr@10": 0.0,
            "ndcg@10": 0.0,
        }

    docs = candidate_doc_ids[qi, order[:top_k]]
    hits = np.array([int(int(d) in rel) for d in docs], dtype=np.int32)
    n_rel = len(rel)

    recall = float(hits.sum() / min(n_rel, top_k))
    success = float(hits.sum() > 0)

    rr = 0.0
    hit_positions = np.flatnonzero(hits)
    if hit_positions.size > 0:
        rr = float(1.0 / (hit_positions[0] + 1))

    dcg = dcg_at_k(hits)
    ideal_hits = np.ones(min(n_rel, top_k), dtype=np.float32)
    idcg = dcg_at_k(ideal_hits)
    ndcg = float(dcg / idcg) if idcg > 0 else 0.0

    return {
        "recall@10": recall,
        "success@10": success,
        "mrr@10": rr,
        "ndcg@10": ndcg,
    }


def aggregate_metric_rows(per_query_rows, name, avg_corrected):
    df = pd.DataFrame(per_query_rows)
    return {
        "strategy": name,
        "queries": int(len(df)),
        "recall@10": float(df["recall@10"].mean()),
        "success@10": float(df["success@10"].mean()),
        "mrr@10": float(df["mrr@10"].mean()),
        "ndcg@10": float(df["ndcg@10"].mean()),
        "avg_corrected_candidates": float(avg_corrected),
    }


per_query_by_depth = {}
fixed_rows = []

for depth in DEPTHS:
    depth = int(depth)
    rows = []
    for qi in range(len(eval_query_rows)):
        rows.append(per_query_metrics_from_order(qi, rank_by_depth[depth][qi], top_k=FINAL_K))
    per_query_by_depth[depth] = pd.DataFrame(rows)
    fixed_rows.append(aggregate_metric_rows(rows, f"always_top{depth}", avg_corrected=depth))

fixed_df = pd.DataFrame(fixed_rows)
fixed_df.to_csv(OUT_DIR / "fixed_depth_strategy_summary.csv", index=False)
display(fixed_df)

## 6. Oracle labels for Top0 / Top20 / Top40 routing

In [ ]:
def build_oracle_labels(primary_metric="recall@10", epsilon=0.0):
    metric_matrix = np.stack(
        [per_query_by_depth[int(d)][primary_metric].to_numpy(dtype=np.float32) for d in DEPTHS],
        axis=1,
    )
    best = metric_matrix.max(axis=1)

    labels = []
    for i in range(metric_matrix.shape[0]):
        ok = np.flatnonzero(metric_matrix[i] >= best[i] - epsilon)
        labels.append(int(DEPTHS[int(ok[0])]))

    return np.asarray(labels, dtype=np.int64), metric_matrix


oracle_labels, oracle_metric_matrix = build_oracle_labels(primary_metric="recall@10", epsilon=0.0)

label_counts = pd.Series(oracle_labels).value_counts().sort_index()
label_dist_df = pd.DataFrame({
    "depth": label_counts.index.astype(int),
    "queries": label_counts.values.astype(int),
    "fraction": label_counts.values / len(oracle_labels),
})
label_dist_df.to_csv(OUT_DIR / "router_oracle_label_distribution.csv", index=False)

oracle_selected_rows = []
for qi, depth in enumerate(oracle_labels):
    oracle_selected_rows.append(per_query_by_depth[int(depth)].iloc[qi].to_dict())

oracle_row = aggregate_metric_rows(
    oracle_selected_rows,
    "oracle_cheapest_best_recall_top0_20_40",
    avg_corrected=float(np.mean(oracle_labels)),
)

oracle_df = pd.DataFrame([oracle_row])
oracle_df.to_csv(OUT_DIR / "router_oracle_summary.csv", index=False)

display(label_dist_df)
display(oracle_df)

## 7. Query-level feature construction

In [ ]:
def safe_entropy_from_scores(scores, temperature=1.0):
    x = scores.astype(np.float64) / temperature
    x = x - np.max(x)
    p = np.exp(x)
    p = p / (p.sum() + 1e-12)
    return float(-(p * np.log(p + 1e-12)).sum())


def count_boundary_crossings(base_scores, corrected_scores, k=10, depth=40):
    base_order = np.argsort(-base_scores)
    boundary = base_scores[base_order[k - 1]]
    outside_topk = set(base_order[k:depth].tolist())
    count = 0
    for j in outside_topk:
        if corrected_scores[j] > boundary:
            count += 1
    return count


def build_query_features():
    rows = []
    base_rank = rank_by_depth[0]
    corr20_rank = rank_by_depth[20]
    corr40_rank = rank_by_depth[40]

    for qi in range(len(eval_query_rows)):
        s = ann_scores[qi]
        c = correction_alpha[qi]
        q = get_eval_query(qi)

        row = {
            "query_index": qi,
            "query_id": eval_query_ids[qi],
        }

        for a, b in [(1, 2), (1, 5), (1, 10), (5, 10), (10, 20), (20, 40), (40, 100)]:
            row[f"gap_s{a}_s{b}"] = float(s[a - 1] - s[b - 1])

        for k in [10, 20, 40, 100]:
            top = s[:k]
            row[f"score_top{k}_mean"] = float(top.mean())
            row[f"score_top{k}_std"] = float(top.std())
            row[f"score_top{k}_range"] = float(top.max() - top.min())
            row[f"score_top{k}_entropy"] = safe_entropy_from_scores(top)

        for k in [10, 20, 40]:
            topc = c[:k]
            row[f"corr_top{k}_mean"] = float(topc.mean())
            row[f"corr_top{k}_std"] = float(topc.std())
            row[f"corr_top{k}_mean_abs"] = float(np.mean(np.abs(topc)))
            row[f"corr_top{k}_max_abs"] = float(np.max(np.abs(topc)))
            row[f"corr_top{k}_pos_frac"] = float(np.mean(topc > 0))
            row[f"corr_top{k}_neg_frac"] = float(np.mean(topc < 0))

        base_corrected20 = score_by_depth[20][qi]
        base_corrected40 = score_by_depth[40][qi]

        row["base_margin_10_11"] = float(s[base_rank[qi, 9]] - s[base_rank[qi, 10]])
        row["base_margin_10_20"] = float(s[base_rank[qi, 9]] - s[base_rank[qi, 19]])
        row["base_margin_10_40"] = float(s[base_rank[qi, 9]] - s[base_rank[qi, 39]])
        row["cross_count_top20_over_base_top10"] = float(count_boundary_crossings(s, base_corrected20, k=10, depth=20))
        row["cross_count_top40_over_base_top10"] = float(count_boundary_crossings(s, base_corrected40, k=10, depth=40))

        row["top10_overlap_base_vs_top20"] = float(len(set(base_rank[qi, :10]) & set(corr20_rank[qi, :10])) / 10)
        row["top10_overlap_base_vs_top40"] = float(len(set(base_rank[qi, :10]) & set(corr40_rank[qi, :10])) / 10)
        row["top20_overlap_base_vs_top40"] = float(len(set(base_rank[qi, :20]) & set(corr40_rank[qi, :20])) / 20)

        exact_error = exact_scores[qi] - ann_scores[qi]
        for k in [10, 20, 40]:
            e = exact_error[:k]
            row[f"offline_exact_error_top{k}_mean"] = float(e.mean())
            row[f"offline_exact_error_top{k}_std"] = float(e.std())
            row[f"offline_exact_error_top{k}_mean_abs"] = float(np.mean(np.abs(e)))
            row[f"offline_corr_error_corr_top{k}"] = float(np.corrcoef(c[:k], e)[0, 1]) if np.std(c[:k]) > 0 and np.std(e) > 0 else 0.0

        row["query_max_abs_dim"] = float(np.max(np.abs(q)))
        row["query_l2_norm"] = float(np.linalg.norm(q))
        row["query_l1_norm"] = float(np.sum(np.abs(q)))
        row["query_dim_std"] = float(np.std(q))

        rows.append(row)

    return pd.DataFrame(rows)


features_df = build_query_features()

feature_cols_all = [
    c for c in features_df.columns
    if c not in {"query_index", "query_id"}
]
feature_cols_deployable = [
    c for c in feature_cols_all
    if not c.startswith("offline_exact_error_") and not c.startswith("offline_corr_error_")
]

features_df.to_csv(OUT_DIR / "router_query_features.csv", index=False)

print("features:", features_df.shape)
print("deployable feature count:", len(feature_cols_deployable))
print("all feature count:", len(feature_cols_all))
display(features_df.head())

## 8. Router models and held-out evaluation

In [ ]:
def make_cv(labels, n_splits=5, random_state=42):
    counts = pd.Series(labels).value_counts()
    if len(counts) > 1 and counts.min() >= n_splits:
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state).split(
            np.zeros(len(labels)), labels
        )
    return KFold(n_splits=n_splits, shuffle=True, random_state=random_state).split(
        np.zeros(len(labels))
    )


def make_models():
    return {
        "logreg_balanced": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                multi_class="auto",
                random_state=RANDOM_STATE,
            ),
        ),
        "random_forest_balanced": RandomForestClassifier(
            n_estimators=400,
            max_depth=5,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.04,
            max_leaf_nodes=8,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        ),
    }


def evaluate_predicted_depths(pred_depths, strategy_name):
    selected_rows = []
    for qi, depth in enumerate(pred_depths):
        selected_rows.append(per_query_by_depth[int(depth)].iloc[qi].to_dict())

    return aggregate_metric_rows(
        selected_rows,
        strategy_name,
        avg_corrected=float(np.mean(pred_depths)),
    )


def run_5fold_router(feature_cols, experiment_name):
    X_feat = features_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
    y = oracle_labels.copy()

    models = make_models()
    summary_rows = []
    prediction_rows = []

    for model_name, model in models.items():
        pred_all = np.zeros_like(y)

        for fold, (train_idx, test_idx) in enumerate(make_cv(y, n_splits=N_SPLITS, random_state=RANDOM_STATE)):
            model.fit(X_feat[train_idx], y[train_idx])
            pred = model.predict(X_feat[test_idx])
            pred_all[test_idx] = pred

            prediction_rows.extend([
                {
                    "experiment": experiment_name,
                    "model": model_name,
                    "fold": int(fold),
                    "query_index": int(qi),
                    "query_id": eval_query_ids[int(qi)],
                    "oracle_depth": int(y[int(qi)]),
                    "pred_depth": int(p),
                }
                for qi, p in zip(test_idx, pred)
            ])

        row = evaluate_predicted_depths(pred_all, f"{experiment_name}:{model_name}")
        row["experiment"] = experiment_name
        row["model"] = model_name
        row["label_accuracy"] = float(accuracy_score(y, pred_all))
        row["label_balanced_accuracy"] = float(balanced_accuracy_score(y, pred_all))
        row["pred_top0_frac"] = float(np.mean(pred_all == 0))
        row["pred_top20_frac"] = float(np.mean(pred_all == 20))
        row["pred_top40_frac"] = float(np.mean(pred_all == 40))
        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(prediction_rows)


deploy_summary_df, deploy_pred_df = run_5fold_router(feature_cols_deployable, "deployable_features")
all_summary_df, all_pred_df = run_5fold_router(feature_cols_all, "offline_exact_proxy_features")

router_summary_df = pd.concat([deploy_summary_df, all_summary_df], ignore_index=True)
router_predictions_df = pd.concat([deploy_pred_df, all_pred_df], ignore_index=True)

router_summary_df.to_csv(OUT_DIR / "router_5fold_summary.csv", index=False)
router_predictions_df.to_csv(OUT_DIR / "router_5fold_predictions.csv", index=False)

display(router_summary_df.sort_values(["recall@10", "avg_corrected_candidates"], ascending=[False, True]))

## 9. Strategy comparison

In [ ]:
strategy_rows = []

strategy_rows.extend(fixed_rows)
strategy_rows.append(oracle_row)

for _, row in router_summary_df.iterrows():
    strategy_rows.append({
        "strategy": row["strategy"],
        "queries": int(row["queries"]),
        "recall@10": float(row["recall@10"]),
        "success@10": float(row["success@10"]),
        "mrr@10": float(row["mrr@10"]),
        "ndcg@10": float(row["ndcg@10"]),
        "avg_corrected_candidates": float(row["avg_corrected_candidates"]),
    })

strategy_df = pd.DataFrame(strategy_rows)
strategy_df["recall_per_corrected_candidate"] = strategy_df["recall@10"] / np.maximum(strategy_df["avg_corrected_candidates"], 1.0)

strategy_df.to_csv(OUT_DIR / "router_strategy_comparison.csv", index=False)

display(strategy_df.sort_values(["recall@10", "avg_corrected_candidates"], ascending=[False, True]))

## 10. Feature importance

In [ ]:
X_dep = features_df[feature_cols_deployable].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
y = oracle_labels.copy()

rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_dep, y)

importance_df = pd.DataFrame({
    "feature": feature_cols_deployable,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df.to_csv(OUT_DIR / "router_feature_importance.csv", index=False)

display(importance_df.head(30))

## 11. Save README and manifest

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


readme = f"""# Learned RARS Router

This package evaluates learned query-adaptive activation for the MS MARCO 1M
Retrieval-Aware Residual Subspace (RARS) sidecar.

## Protocol

- Dataset: MS MARCO deterministic 1M passage subset
- Queries: {len(eval_query_rows)}
- Base index: frozen IVF-PQ `M=32`, `nlist=512`, `nprobe=16`
- Candidate pool: Top-{TOP_L}
- Sidecar: `{BASIS_NAME}` rank-{RANK} int8 RARS correction
- Correction alpha: `{ALPHA}`
- Candidate depths: Top0 / Top20 / Top40
- Router evaluation: {N_SPLITS}-fold held-out query-level routing

## Outputs

- `fixed_depth_strategy_summary.csv`
- `router_oracle_label_distribution.csv`
- `router_oracle_summary.csv`
- `router_query_features.csv`
- `router_5fold_summary.csv`
- `router_5fold_predictions.csv`
- `router_strategy_comparison.csv`
- `router_feature_importance.csv`
- `router_config.json`
- `manifest.json`

## Interpretation rule

A learned router should only be reported as useful if it improves the cost-quality
trade-off against fixed-depth baselines on held-out folds.

Conservative reporting rule:

- If a learned router approaches Always Top40 quality with much lower average
  correction depth, report it as useful query-adaptive activation.
- If it does not beat fixed Top20 or fixed Top40 under a meaningful cost-quality
  comparison, keep fixed Top20 as the strongest deployable cost-aware point and
  report the learned router as a negative diagnostic.
"""

with open(OUT_DIR / "README.md", "w") as f:
    f.write(readme)

artifact_names = [
    "README.md",
    "router_config.json",
    "fixed_depth_strategy_summary.csv",
    "router_oracle_label_distribution.csv",
    "router_oracle_summary.csv",
    "router_query_features.csv",
    "router_5fold_summary.csv",
    "router_5fold_predictions.csv",
    "router_strategy_comparison.csv",
    "router_feature_importance.csv",
]

manifest = {
    "package": "learned_rars_router_msmarco_1m",
    "config": CONFIG,
    "artifacts": {},
}

for name in artifact_names:
    path = OUT_DIR / name
    if path.exists():
        manifest["artifacts"][name] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("saved package:", OUT_DIR)
print(json.dumps(manifest, indent=2)[:3000])

## 12. How to read the result

Compare:

```text
always_top20
always_top40
oracle_cheapest_best_recall_top0_20_40
deployable_features:<model>
offline_exact_proxy_features:<model>
```

The most important columns are:

```text
recall@10
mrr@10
ndcg@10
avg_corrected_candidates
```

A good result is not just high Recall@10. It should either:

1. approach Always Top40 quality with much lower average correction depth, or
2. beat Always Top20 at comparable correction depth.

If neither happens, the correct conclusion is that oracle headroom exists, but
the current learned router is not robust enough; fixed Top20 remains the best
deployable cost-aware operating point.